# Carbon-Aware Scheduler Parameter Sweep Analysis

This notebook runs and analyzes multiple carbon-aware scheduler configurations to identify optimal parameters for carbon reduction.

## Parameters Being Tested:
- **carbonDelayThreshold**: Percentile threshold for "high carbon" (lower = more aggressive delays)
- **maxDelayHours**: Maximum time to delay a task
- **forecastHorizon**: How far ahead to look in carbon forecast
- **slackThresholdMultiplier**: Safety margin for deadline compliance
- **prioritizeCriticalPath**: Enable workflow-aware heuristics

In [ ]:
import subprocess
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Paths
BASE_DIR = Path('/home/energyless')
RUNNER_PATH = BASE_DIR / 'OpenDCExperimentRunner/bin/OpenDCExperimentRunner'
EXPERIMENTS_DIR = BASE_DIR / 'scheduler_comparison_experiments'
OUTPUT_DIR = BASE_DIR / 'output'
RESULTS_DIR = EXPERIMENTS_DIR / 'parameter_sweep_results'
RESULTS_DIR.mkdir(exist_ok=True)

print("✅ Environment configured")

## Step 1: Define Parameter Sweep Configurations

We'll test 10 different configurations ranging from conservative to very aggressive carbon optimization.

In [ ]:
# Base experiment template
BASE_CONFIG = {
    "name": "carbon_aware_sweep",
    "topologies": [{"pathToFile": "input/topologies/sample_NL_small.json"}],
    "workloads": [{
        "pathToFile": "input/synthetic_traces/carbon_test_long_n80_edge_prob0.3_seed1_deadline_default",
        "type": "ComputeWorkload"
    }],
    "allocationPolicies": [{
        "type": "carbonAware",
        "filters": [
            {"type": "Compute"},
            {"type": "VCpu", "allocationRatio": 1.0},
            {"type": "Ram", "allocationRatio": 1.0}
        ],
        "weighers": [{"type": "Ram", "multiplier": 1.0}],
        "subsetSize": 1
    }],
    "exportModels": [{
        "exportInterval": 3600,
        "printFrequency": 24,
        "filesToExport": ["host", "powerSource", "service", "task"]
    }]
}

# Parameter configurations to test
SWEEP_CONFIGS = [
    # Current baseline
    {"name": "baseline_current", "carbonDelayThreshold": 0.2, "maxDelayHours": 8, 
     "forecastHorizon": 48, "slackThresholdMultiplier": 1.5, "prioritizeCriticalPath": False},
    
    # Aggressive carbon optimization
    {"name": "aggressive_low_threshold", "carbonDelayThreshold": 0.1, "maxDelayHours": 12,
     "forecastHorizon": 72, "slackThresholdMultiplier": 2.0, "prioritizeCriticalPath": True},
    
    # Conservative safety-first
    {"name": "conservative_high_safety", "carbonDelayThreshold": 0.3, "maxDelayHours": 4,
     "forecastHorizon": 24, "slackThresholdMultiplier": 2.5, "prioritizeCriticalPath": True},
    
    # Balanced workflow-aware
    {"name": "balanced_workflow_aware", "carbonDelayThreshold": 0.2, "maxDelayHours": 8,
     "forecastHorizon": 48, "slackThresholdMultiplier": 2.0, "prioritizeCriticalPath": True},
    
    # Long forecast horizon
    {"name": "long_horizon", "carbonDelayThreshold": 0.15, "maxDelayHours": 10,
     "forecastHorizon": 72, "slackThresholdMultiplier": 2.0, "prioritizeCriticalPath": True},
    
    # Short horizon reactive
    {"name": "short_horizon_reactive", "carbonDelayThreshold": 0.25, "maxDelayHours": 6,
     "forecastHorizon": 12, "slackThresholdMultiplier": 1.8, "prioritizeCriticalPath": True},
    
    # Very aggressive
    {"name": "very_aggressive", "carbonDelayThreshold": 0.05, "maxDelayHours": 16,
     "forecastHorizon": 96, "slackThresholdMultiplier": 2.5, "prioritizeCriticalPath": True},
    
    # Medium slack workflow
    {"name": "medium_slack_workflow", "carbonDelayThreshold": 0.2, "maxDelayHours": 8,
     "forecastHorizon": 48, "slackThresholdMultiplier": 1.8, "prioritizeCriticalPath": True},
    
    # Tight threshold long delay
    {"name": "tight_threshold_long_delay", "carbonDelayThreshold": 0.15, "maxDelayHours": 12,
     "forecastHorizon": 60, "slackThresholdMultiplier": 2.2, "prioritizeCriticalPath": True},
    
    # No workflow awareness
    {"name": "no_workflow_awareness", "carbonDelayThreshold": 0.2, "maxDelayHours": 8,
     "forecastHorizon": 48, "slackThresholdMultiplier": 1.5, "prioritizeCriticalPath": False}
]

print(f"✅ Configured {len(SWEEP_CONFIGS)} parameter combinations to test")
print("\nConfigurations:")
for i, config in enumerate(SWEEP_CONFIGS, 1):
    print(f"{i:2d}. {config['name']}")

## Step 2: Helper Functions

In [ ]:
def create_experiment_config(config_params):
    """Create experiment JSON with given parameters."""
    config = BASE_CONFIG.copy()
    config["name"] = f"carbon_aware_sweep_{config_params['name']}"
    
    policy = config["allocationPolicies"][0].copy()
    policy.update({
        "carbonDelayThreshold": config_params["carbonDelayThreshold"],
        "maxDelayHours": config_params["maxDelayHours"],
        "forecastHorizon": config_params["forecastHorizon"],
        "slackThresholdMultiplier": config_params["slackThresholdMultiplier"],
        "prioritizeCriticalPath": config_params["prioritizeCriticalPath"]
    })
    config["allocationPolicies"][0] = policy
    return config

def run_experiment(config_params, config_num, total_configs):
    """Run a single experiment."""
    config = create_experiment_config(config_params)
    exp_name = config_params['name']
    
    # Save config
    config_file = EXPERIMENTS_DIR / f"sweep_{exp_name}.json"
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"\n{'='*80}")
    print(f"Experiment {config_num}/{total_configs}: {exp_name}")
    print(f"{'='*80}")
    
    # Run experiment
    result = subprocess.run(
        [str(RUNNER_PATH), "--experiment-path", str(config_file)],
        capture_output=True, text=True, cwd=str(BASE_DIR)
    )
    
    if result.returncode != 0:
        print(f"❌ Failed: {result.stderr}")
        return None
    
    print(f"✅ Completed!")
    
    # Parse results
    output_path = OUTPUT_DIR / f"carbon_aware_sweep_{exp_name}" / "raw-output/0/seed=0"
    try:
        return parse_results(output_path, exp_name, config_params)
    except Exception as e:
        print(f"❌ Parse error: {e}")
        return None

def parse_results(output_path, exp_name, config_params):
    """Parse experiment results."""
    power = pd.read_parquet(output_path / "powerSource.parquet")
    task = pd.read_parquet(output_path / "task.parquet")
    host = pd.read_parquet(output_path / "host.parquet")
    
    return {
        'experiment': exp_name,
        'carbonDelayThreshold': config_params['carbonDelayThreshold'],
        'maxDelayHours': config_params['maxDelayHours'],
        'forecastHorizon': config_params['forecastHorizon'],
        'slackThresholdMultiplier': config_params['slackThresholdMultiplier'],
        'prioritizeCriticalPath': config_params['prioritizeCriticalPath'],
        'total_carbon_kg': power['carbon_emission'].sum(),
        'total_energy_kwh': power['energy_usage'].sum() / (1000 * 1000),
        'makespan_hours': task['finish_time'].max() / (1000 * 3600),
        'avg_wait_time_hours': task['scheduling_delay'].mean() / (1000 * 3600),
        'max_wait_time_hours': task['scheduling_delay'].max() / (1000 * 3600),
        'completed_tasks': len(task[task['task_state'] == 'COMPLETED']),
        'total_tasks': len(task),
        'avg_cpu_util': host['cpu_utilization'].mean() * 100
    }

print("✅ Helper functions defined")

## Step 3: Run Parameter Sweep

⚠️ **This will take some time!** Each experiment may take 2-5 minutes.

Estimated total time: ~20-50 minutes for all 10 configurations.

In [ ]:
print("🚀 Starting Parameter Sweep...")
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

all_metrics = []

for i, config in enumerate(SWEEP_CONFIGS, 1):
    metrics = run_experiment(config, i, len(SWEEP_CONFIGS))
    if metrics:
        all_metrics.append(metrics)

# Create results DataFrame
df_results = pd.DataFrame(all_metrics)

# Save results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_file = RESULTS_DIR / f"sweep_results_{timestamp}.csv"
df_results.to_csv(csv_file, index=False)

print(f"\n{'='*80}")
print(f"✅ Sweep completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✅ Results saved to: {csv_file}")
print(f"{'='*80}")

## Step 4: Analyze Results

Sort configurations by carbon emissions to find the best performer.

In [ ]:
# Sort by carbon emissions
df_sorted = df_results.sort_values('total_carbon_kg')

print("\n🏆 RANKING BY CARBON EMISSIONS (Lower is Better)")
print("="*100)

display_cols = ['experiment', 'total_carbon_kg', 'makespan_hours', 'avg_wait_time_hours',
                'carbonDelayThreshold', 'maxDelayHours', 'forecastHorizon', 'prioritizeCriticalPath']

print(df_sorted[display_cols].to_string(index=False))

# Calculate improvements vs baseline
baseline = df_results[df_results['experiment'] == 'baseline_current'].iloc[0]

print("\n\n📊 IMPROVEMENTS VS BASELINE")
print("="*100)

for _, row in df_sorted.iterrows():
    if row['experiment'] == 'baseline_current':
        continue
    
    carbon_imp = ((baseline['total_carbon_kg'] - row['total_carbon_kg']) / baseline['total_carbon_kg']) * 100
    makespan_chg = ((row['makespan_hours'] - baseline['makespan_hours']) / baseline['makespan_hours']) * 100
    
    print(f"\n{row['experiment']}:")
    print(f"  Carbon: {carbon_imp:+.2f}% | Makespan: {makespan_chg:+.2f}%")

## Step 5: Visualizations

In [ ]:
# Carbon vs Makespan tradeoff
fig, ax = plt.subplots(figsize=(14, 8))

scatter = ax.scatter(df_results['total_carbon_kg'], df_results['makespan_hours'],
                     s=200, alpha=0.6, c=df_results['maxDelayHours'], cmap='viridis')

# Annotate points
for _, row in df_results.iterrows():
    ax.annotate(row['experiment'], 
                (row['total_carbon_kg'], row['makespan_hours']),
                fontsize=8, ha='right', va='bottom')

ax.set_xlabel('Total Carbon Emissions (kg CO2)', fontsize=12, weight='bold')
ax.set_ylabel('Makespan (hours)', fontsize=12, weight='bold')
ax.set_title('Carbon vs Makespan Tradeoff\n(Color = Max Delay Hours)', 
             fontsize=14, weight='bold')
plt.colorbar(scatter, ax=ax, label='Max Delay Hours')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Carbon emissions
df_sorted.plot(x='experiment', y='total_carbon_kg', kind='bar', ax=ax1, 
               color='green', alpha=0.7, legend=False)
ax1.set_title('Carbon Emissions by Configuration', fontsize=13, weight='bold')
ax1.set_ylabel('Total Carbon (kg CO2)', fontsize=11)
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=45, labelsize=9)
ax1.grid(axis='y', alpha=0.3)

# Makespan
df_sorted.plot(x='experiment', y='makespan_hours', kind='bar', ax=ax2,
               color='blue', alpha=0.7, legend=False)
ax2.set_title('Makespan by Configuration', fontsize=13, weight='bold')
ax2.set_ylabel('Makespan (hours)', fontsize=11)
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=45, labelsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of parameter impact
fig, ax = plt.subplots(figsize=(12, 8))

# Create pivot for heatmap (carbon by threshold and max delay)
pivot_data = df_results.pivot_table(
    values='total_carbon_kg',
    index='carbonDelayThreshold',
    columns='maxDelayHours',
    aggfunc='mean'
)

sns.heatmap(pivot_data, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax, cbar_kws={'label': 'Carbon (kg)'})
ax.set_title('Carbon Emissions: Threshold vs Max Delay Hours', fontsize=14, weight='bold')
ax.set_xlabel('Max Delay Hours', fontsize=11)
ax.set_ylabel('Carbon Delay Threshold', fontsize=11)
plt.tight_layout()
plt.show()

## Step 6: Best Configuration Recommendation

In [ ]:
best = df_sorted.iloc[0]

print("\n" + "="*80)
print("🏆 RECOMMENDED CONFIGURATION (Lowest Carbon)")
print("="*80)
print(f"\nName: {best['experiment']}")
print(f"\nMetrics:")
print(f"  Total Carbon: {best['total_carbon_kg']:.2f} kg CO2")
print(f"  Makespan: {best['makespan_hours']:.2f} hours")
print(f"  Avg Wait Time: {best['avg_wait_time_hours']:.2f} hours")

carbon_imp = ((baseline['total_carbon_kg'] - best['total_carbon_kg']) / baseline['total_carbon_kg']) * 100
print(f"\n  Carbon Improvement vs Baseline: {carbon_imp:+.2f}%")

print(f"\nOptimal Parameters:")
print(f"  carbonDelayThreshold: {best['carbonDelayThreshold']}")
print(f"  maxDelayHours: {best['maxDelayHours']}")
print(f"  forecastHorizon: {best['forecastHorizon']}")
print(f"  slackThresholdMultiplier: {best['slackThresholdMultiplier']}")
print(f"  prioritizeCriticalPath: {best['prioritizeCriticalPath']}")

# Save optimal config
optimal_config = create_experiment_config({
    'name': 'optimal',
    'carbonDelayThreshold': best['carbonDelayThreshold'],
    'maxDelayHours': int(best['maxDelayHours']),
    'forecastHorizon': int(best['forecastHorizon']),
    'slackThresholdMultiplier': best['slackThresholdMultiplier'],
    'prioritizeCriticalPath': bool(best['prioritizeCriticalPath'])
})

optimal_file = EXPERIMENTS_DIR / 'carbon_aware_optimal.json'
with open(optimal_file, 'w') as f:
    json.dump(optimal_config, f, indent=2)

print(f"\n✅ Optimal configuration saved to: {optimal_file}")
print("="*80)